# torch `nn.Embedding`

## 사전학습된 임베딩을 사용하지 않는 경우

In [1]:
# 감성 분석 샘플 데이터
sentences = [
    'nice great best amazing',
    'stop lies',
    'pitiful nerd',
    'excellent work',
    'supreme quality',
    'bad',
    'highly respectable'
]
labels = [1, 0, 0, 1, 1, 0, 1]

In [2]:
# 토큰화
from nltk.tokenize import word_tokenize

tokenized_sentences = [word_tokenize(sent) for sent in sentences]
tokenized_sentences

[['nice', 'great', 'best', 'amazing'],
 ['stop', 'lies'],
 ['pitiful', 'nerd'],
 ['excellent', 'work'],
 ['supreme', 'quality'],
 ['bad'],
 ['highly', 'respectable']]

In [5]:
# 단어 사전/정수 인코딩
from collections import Counter

tokens = [token for sent in tokenized_sentences for token in sent]
word_counts = Counter(tokens)
print(word_counts)

word_to_index = {word: index + 2 for index, word in enumerate(tokens)}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1
word_to_index = dict(sorted(word_to_index.items(), key=lambda x:x[1]))
print(word_to_index)

vocab_size = len(word_to_index)
vocab_size

Counter({'nice': 1, 'great': 1, 'best': 1, 'amazing': 1, 'stop': 1, 'lies': 1, 'pitiful': 1, 'nerd': 1, 'excellent': 1, 'work': 1, 'supreme': 1, 'quality': 1, 'bad': 1, 'highly': 1, 'respectable': 1})
{'<PAD>': 0, '<UNK>': 1, 'nice': 2, 'great': 3, 'best': 4, 'amazing': 5, 'stop': 6, 'lies': 7, 'pitiful': 8, 'nerd': 9, 'excellent': 10, 'work': 11, 'supreme': 12, 'quality': 13, 'bad': 14, 'highly': 15, 'respectable': 16}


17

In [8]:
def texts_to_sequences(sentences, word_to_index):

    sequneces = []

    for sent in sentences:
        sequnece = []
        for token in sent:
            if token in word_to_index:
                sequnece.append(word_to_index[token])
            else:
                sequnece.append(word_to_index['<UNK>'])
        
        sequneces.append(sequnece)

    return sequneces

sequences = texts_to_sequences(tokenized_sentences, word_to_index)
sequences

[[2, 3, 4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14], [15, 16]]

In [9]:
# 패딩
import numpy as np

def pad_sequences(sequences, maxlen):
    padded_sqeunces = np.zeros((len(sequences), maxlen), dtype=int)
    for index, seq in enumerate(sequences):
        padded_sqeunces[index, :len(seq)] = seq[:maxlen]
    return padded_sqeunces

padded_sequences = pad_sequences(sequences, maxlen=4)
padded_sequences

array([[ 2,  3,  4,  5],
       [ 6,  7,  0,  0],
       [ 8,  9,  0,  0],
       [10, 11,  0,  0],
       [12, 13,  0,  0],
       [14,  0,  0,  0],
       [15, 16,  0,  0]])

In [10]:
# 모델 생성
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class SimpleNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0   # 패딩 벡터는 학습하지 않음
        )
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        _, h_n = self.rnn(embedded)
        out = self.out(h_n.squeeze(0))
        return out
    
embedding_dim = 100
model = SimpleNet(vocab_size, embedding_dim, hidden_size=8)
model

SimpleNet(
  (embedding): Embedding(17, 100, padding_idx=0)
  (rnn): RNN(100, 8, batch_first=True)
  (out): Linear(in_features=8, out_features=1, bias=True)
)

In [11]:
import pandas as pd 

# 학습 전 임베딩 벡터
wv = model.embedding.weight.data
print(wv.shape)

# 특정 단어 벡터
vocab = word_to_index.keys()
pd.DataFrame(wv, index=vocab)

torch.Size([17, 100])


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,1.828872,0.509676,-0.288132,-0.885471,-0.136452,-1.729298,0.109523,1.235304,-1.063755,-0.312190,-0.594607,0.092759,0.649567,1.426396,-0.474850,-1.126105,0.994402,1.081112,0.412258,-0.395440,-0.655790,0.072337,-1.061913,1.375735,-1.403142,-1.270225,1.573803,-0.574792,1.281632,-1.390553,-2.119517,-1.352808,-0.824620,-1.040872,0.875839,0.971977,0.986451,0.764580,0.209368,-0.412079,...,-0.323610,-0.000344,-0.139496,0.926068,-1.175863,-0.369505,-0.094491,0.939226,0.097257,0.908592,0.825777,-0.743451,-0.950839,-0.801201,-0.489871,-1.054415,1.079278,-0.015614,-0.257981,-1.058287,1.836445,-0.687306,-1.536166,-0.785671,-1.713296,-0.373460,-0.355808,-0.956215,0.103779,-0.317353,-0.691616,1.013262,-2.017559,-0.654979,-1.547667,-0.233359,-1.520748,-1.140529,0.947225,0.263063
nice,-0.477682,-0.324669,0.173214,-0.538858,-0.458331,0.541922,-1.152541,0.481452,0.176531,-0.097617,0.393983,0.375201,-0.404486,-1.186242,1.487936,0.846909,-0.484867,1.202305,-1.549451,0.064322,1.937630,-0.367925,0.100045,-0.160876,1.341432,-0.714438,-1.166301,-0.210041,0.617613,1.229303,0.195261,0.275735,1.095366,0.263872,-1.013343,-0.072279,1.530655,0.249429,-1.715058,0.467196,...,1.556279,-1.703555,-0.332274,1.398893,-1.623519,-0.724089,-1.013712,-0.688291,0.016659,0.158300,1.207098,-0.742849,-1.796431,0.295639,-0.852303,-0.771214,0.133697,-1.738723,-0.035874,0.168961,-1.038753,0.767301,-0.648722,1.110077,-0.014347,0.416964,0.034695,-0.316741,0.775976,-0.902386,0.099572,1.205805,-0.194539,1.832150,-0.881837,-1.430561,2.852673,0.523289,-0.335612,-0.443575
great,0.744822,0.027574,-0.421996,0.664136,0.217603,-0.278037,-0.795439,0.180476,0.019993,-1.933524,1.058786,-0.312400,-1.943383,1.919461,0.510896,-0.789249,-1.163831,-1.433409,-0.457456,-0.176304,0.053859,1.193183,-0.527590,-1.571307,0.199437,0.020886,1.025060,1.686123,-2.038705,-0.275310,-1.008551,-0.447037,-0.079229,0.778536,1.322134,-0.990268,-2.707922,0.850177,-0.046963,0.174397,...,0.285783,0.057848,0.848210,1.292331,1.580662,-2.597152,1.229164,1.731627,-0.181210,-0.478205,0.461151,0.816227,2.025101,-0.933082,-0.804089,-0.639116,-0.229780,0.642480,-0.753551,-1.755104,-0.260220,-0.050427,0.074275,-1.775881,1.801078,0.465831,-0.390366,-0.379815,0.324506,1.458271,0.503386,-0.718428,-1.725051,-0.855164,0.090429,0.064716,-0.014671,-1.283377,1.294117,-0.394892
best,0.376507,-0.120035,-0.166980,-1.824778,0.930563,-0.299263,0.110726,-0.870323,-1.444145,1.088844,-0.788031,0.497170,0.233468,-1.534737,1.339572,0.346668,0.968150,-0.198242,-0.093646,-1.070608,0.786517,-0.919904,0.685774,0.672766,1.525054,0.564517,-0.826995,-0.085094,-0.294761,1.047329,-1.164087,-0.273536,-1.382159,-1.590091,0.348362,0.550049,-0.755648,-0.489870,-0.243128,-1.672175,...,-0.251557,0.086661,-0.882450,1.159970,-0.471715,0.446918,-1.034273,-1.128167,0.273230,-0.824000,-0.222952,-0.121700,-0.618948,-1.265465,-0.397759,0.134316,-0.371238,-0.802541,-0.840353,-0.931984,-1.303029,0.362206,-2.580079,-0.261928,-0.137816,1.374646,-0.325825,-1.117623,-0.486130,-0.174182,1.125312,-0.621268,-0.731141,-

In [13]:
# 모델 학습
X = torch.tensor(padded_sequences, dtype=torch.long)            # (batch_size, seq_len)
y = torch.tensor(labels, dtype=torch.float).unsqueeze(1)        # (batch_size, 1)

print(f'{X.shape=}, {y.shape=}')

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

X.shape=torch.Size([7, 4]), y.shape=torch.Size([7, 1])


In [14]:
for epoch in range(20):

    epoch_loss = 0

    for x_batch, y_batch in dataloader:

        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f'Epoch {epoch+1} : Loss {epoch_loss / len(dataloader)}')


Epoch 1 : Loss 0.6598442494869232
Epoch 2 : Loss 0.5655427277088165
Epoch 3 : Loss 0.542184442281723
Epoch 4 : Loss 0.47659873217344284
Epoch 5 : Loss 0.45346350967884064
Epoch 6 : Loss 0.3899667337536812
Epoch 7 : Loss 0.3422394469380379
Epoch 8 : Loss 0.307168647646904
Epoch 9 : Loss 0.2530880756676197
Epoch 10 : Loss 0.21902861818671227
Epoch 11 : Loss 0.18650979921221733
Epoch 12 : Loss 0.15189366042613983
Epoch 13 : Loss 0.1311190463602543
Epoch 14 : Loss 0.12752268835902214
Epoch 15 : Loss 0.09968273714184761
Epoch 16 : Loss 0.09263269975781441
Epoch 17 : Loss 0.0851017776876688
Epoch 18 : Loss 0.06866804324090481
Epoch 19 : Loss 0.06136411055922508
Epoch 20 : Loss 0.058747841976583004


In [15]:
# 평가/예측
model.eval()
with torch.no_grad():
    output = model(X)
    prob = torch.sigmoid(output)
    pred = (prob >= 0.5).int()

print(labels)
print(pred.squeeze().detach().numpy())

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]


In [16]:
# 학습 후 임베딩 벡터
wv = model.embedding.weight.data
print(wv.shape)

# 특정 단어 벡터
vocab = word_to_index.keys()
pd.DataFrame(wv, index=vocab)

torch.Size([17, 100])


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,1.828872,0.509676,-0.288132,-0.885471,-0.136452,-1.729298,0.109523,1.235304,-1.063755,-0.312190,-0.594607,0.092759,0.649567,1.426396,-0.474850,-1.126105,0.994402,1.081112,0.412258,-0.395440,-0.655790,0.072337,-1.061913,1.375735,-1.403142,-1.270225,1.573803,-0.574792,1.281632,-1.390553,-2.119517,-1.352808,-0.824620,-1.040872,0.875839,0.971977,0.986451,0.764580,0.209368,-0.412079,...,-0.323610,-0.000344,-0.139496,0.926068,-1.175863,-0.369505,-0.094491,0.939226,0.097257,0.908592,0.825777,-0.743451,-0.950839,-0.801201,-0.489871,-1.054415,1.079278,-0.015614,-0.257981,-1.058287,1.836445,-0.687306,-1.536166,-0.785671,-1.713296,-0.373460,-0.355808,-0.956215,0.103779,-0.317353,-0.691616,1.013262,-2.017559,-0.654979,-1.547667,-0.233359,-1.520748,-1.140529,0.947225,0.263063
nice,-0.431791,-0.344950,0.139104,-0.588089,-0.477384,0.597906,-1.181501,0.494113,0.134472,-0.155419,0.366662,0.345435,-0.355005,-1.153522,1.473365,0.874779,-0.534234,1.217018,-1.502377,0.090156,1.993287,-0.348216,0.044873,-0.099540,1.426827,-0.671856,-1.159387,-0.257039,0.620819,1.221506,0.176175,0.310099,1.055489,0.267081,-1.045566,-0.041330,1.473599,0.262862,-1.748304,0.430407,...,1.589894,-1.683581,-0.374460,1.359831,-1.637397,-0.677515,-1.040912,-0.656625,-0.014630,0.178162,1.216384,-0.703257,-1.828612,0.302665,-0.895232,-0.741600,0.111720,-1.699978,-0.066724,0.160837,-0.990432,0.754047,-0.679407,1.113857,0.021711,0.446429,0.030372,-0.380290,0.735086,-0.899837,0.060870,1.229727,-0.210185,1.799867,-0.859979,-1.522640,2.892916,0.479077,-0.298619,-0.385998
great,0.775211,-0.007089,-0.465157,0.625159,0.163575,-0.283193,-0.831078,0.211194,-0.008984,-1.927214,1.080156,-0.309517,-1.956683,1.959296,0.466524,-0.840631,-1.123537,-1.463184,-0.504572,-0.161354,0.080877,1.246950,-0.561669,-1.551920,0.172600,0.073397,1.074471,1.674849,-1.985889,-0.334711,-1.038363,-0.475374,-0.132601,0.818754,1.287918,-0.952602,-2.652296,0.866497,-0.102744,0.208378,...,0.323549,0.095379,0.872946,1.262531,1.573483,-2.568551,1.173046,1.792692,-0.177413,-0.435674,0.506322,0.873472,1.963316,-0.979358,-0.801927,-0.588126,-0.289284,0.626617,-0.794986,-1.727049,-0.246596,-0.101139,0.035315,-1.834601,1.854056,0.509073,-0.346267,-0.388798,0.366666,1.499782,0.463987,-0.696435,-1.747282,-0.894223,0.138816,0.093442,0.010474,-1.315955,1.341755,-0.420812
best,0.409805,-0.156184,-0.134265,-1.820902,0.958444,-0.239459,0.087524,-0.930757,-1.488958,1.032092,-0.759424,0.443903,0.228951,-1.501552,1.310804,0.318117,0.992067,-0.152750,-0.124546,-1.101909,0.823522,-0.897075,0.649890,0.713584,1.493922,0.595152,-0.802957,-0.071942,-0.266581,1.030335,-1.119569,-0.308944,-1.435625,-1.553598,0.385282,0.576657,-0.732084,-0.533682,-0.218283,-1.714388,...,-0.222379,0.049725,-0.850060,1.104138,-0.423908,0.486215,-0.979405,-1.165240,0.235256,-0.788818,-0.197686,-0.099831,-0.591163,-1.291421,-0.376970,0.171901,-0.375680,-0.842025,-0.864977,-0.981099,-1.276890,0.336015,-2.620743,-0.276110,-0.158961,1.427118,-0.291119,-1.154519,-0.454627,-0.221506,1.144827,-0.657247,-0.693769,

## 사전학습된 임베딩을 사용하는 경우

https://code.google.com/archive/p/word2vec/

In [18]:
# 사전 학습 된 임베딩 로드
from gensim.models import KeyedVectors

model_wv = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary=True)
model_wv.vectors.shape

(3000000, 300)

In [ ]:
print(model_wv.most_similar('man', topn=5))
print(model_wv.most_similar('woman', topn=5))
print(model_wv.most_similar('king', topn=5))
print(model_wv.most_similar('queen', topn=5))

[('woman', 0.7664012908935547), ('boy', 0.6824871301651001), ('teenager', 0.6586930155754089), ('teenage_girl', 0.6147903203964233), ('girl', 0.5921714305877686)]
[('man', 0.7664012908935547), ('girl', 0.7494640946388245), ('teenage_girl', 0.7336829304695129), ('teenager', 0.6317085027694702), ('lady', 0.6288785934448242)]
[('kings', 0.7138045430183411), ('queen', 0.6510956883430481), ('monarch', 0.6413194537162781), ('crown_prince', 0.6204220056533813), ('prince', 0.6159993410110474)]
[('queens', 0.739944338798523), ('princess', 0.7070532441139221), ('king', 0.6510956883430481), ('monarch', 0.6383602023124695), ('very_pampered_McElhatton', 0.6357026696205139)]


In [20]:
print(model_wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=5))
print(model_wv.most_similar(positive=['Paris', 'Italy'], negative=['France'], topn=5))

[('queen', 0.7118193507194519), ('monarch', 0.6189674139022827), ('princess', 0.5902431011199951), ('crown_prince', 0.5499460697174072), ('prince', 0.5377321839332581)]
[('Milan', 0.7222140431404114), ('Rome', 0.7028310298919678), ('Palermo_Sicily', 0.5967570543289185), ('Italian', 0.5911272764205933), ('Tuscany', 0.5632812976837158)]


In [22]:
# 모델의 Embedding layer에 초기화 할 embedding_matrix
print(len(word_to_index))   # (17, 300) 크기의 embedding_matrix 생성

embedding_matrix = np.zeros((len(word_to_index), model_wv.vectors.shape[1]))
print(embedding_matrix.shape)

17
(17, 300)


In [23]:
def get_word_embedding(word):
    if word in model_wv:
        return model_wv[word]
    else:
        return None
    
for word, index in word_to_index.items():
    if index >= 2:
        emb = get_word_embedding(word)
        if emb is not None:
            embedding_matrix[index] = emb


In [24]:
pd.DataFrame(embedding_matrix, index=word_to_index.keys())

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
nice,0.158203,0.105957,-0.189453,0.386719,0.083496,-0.267578,0.083496,0.113281,-0.104004,0.178711,-0.123535,-0.222656,-0.018066,-0.253906,0.131836,0.085938,0.161133,0.110840,-0.110840,-0.085938,0.026733,0.345703,0.151367,-0.004150,0.104980,0.049072,-0.069824,0.086426,0.031982,-0.028442,-0.157227,0.118652,0.361328,0.001732,0.052979,-0.234375,0.117676,0.086426,-0.011230,0.259766,...,-0.081055,-0.066895,-0.318359,-0.084473,0.135742,0.062500,0.070801,-0.142578,-0.112793,0.014526,-0.066895,0.038818,0.194336,0.095215,0.113770,-0.124512,0.137695,-0.188477,-0.052246,0.158203,0.098633,-0.043701,-0.060547,0.216797,0.040771,-0.146484,-0.189453,-0.251953,-0.168945,-0.086426,-0.085449,0.189453,-0.146484,0.134766,-0.040771,0.032715,0.089355,-0.267578,0.008362,-0.213867
great,0.071777,0.208008,-0.028442,0.178711,0.132812,-0.099609,0.096191,-0.116699,-0.008545,0.148438,-0.033447,-0.185547,0.041016,-0.089844,0.021729,0.069336,0.180664,0.222656,-0.100586,-0.069336,0.000104,0.160156,0.040771,0.073730,0.153320,0.067871,-0.103027,0.041748,0.042725,-0.110352,-0.066895,0.041992,0.250000,0.212891,0.159180,0.014465,-0.048828,0.013977,0.003555,0.209961,...,-0.068359,-0.139648,-0.159180,-0.017944,0.021240,0.073730,0.130859,-0.080566,0.029907,0.015564,-0.166016,0.150391,-0.006775,0.010132,0.114746,-0.148438,-0.045898,-0.139648,-0.173828,-0.042725,-0.058105,0.052246,-0.111328,0.084473,-0.025513,0.140625,-0.181641,0.017212,-0.137695,-0.014771,-0.011475,0.064453,-0.289062,-0.048096,-0.199219,-0.071289,0.064453,-0.167969,-0.020874,-0.142578
best,-0.126953,0.021973,0.287109,0.153320,0.127930,0.032715,-0.115723,-0.029541,0.153320,0.011292,0.139648,-0.086914,0.257812,0.073730,-0.018921,0.125000,0.090820,-0.001556,-0.031982,-0.145508,0.047607,0.173828,-0.146484,0.006012,0.030273,0.040771,-0.066406,0.184570,0.097168,-0.104980,0.024902,0.056396,0.165039,0.090820,0.185547,0.225586,-0.039795,-0.167969,-0.069336,0.019653,...,-0.178711,0.120605,-0.035889,0.095703,0.152344,0.003998,-0.059082,-0.032471,-0.054199,-0.005493,-0.045654,-0.001526,-0.050293,0.255859,0.048340,-0.019409,-0.127930,-0.088379,-0.225586,0.087402,0.205078,0.085938,0.066406,0.108398,-0.191406,0.070312,-0.163086,-0.002472,0.020264,0.001701,0.006439,-0.033936,-0.166016,-0.016846,-0.048584,-0.022827,-0